In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset

from dataset_generator import MedicalSequenceDataset
from sequence_generator import MedicalSequenceGenerator, MedicalSequence
from transisition_model import TransitionModel
from utils.gpu_check import get_device

In [ ]:
device, CUDA = get_device()

In [2]:
N_CABINETS = 5

start_event_probs = [0.4, 0.3, 0.2, 0.1, 0.0]

cab_event_matrix = [
    [0.00, 0.3, 0.4, 0.15, 0.15],
    [0.15, 0.00, 0.4, 0.35, 0.1],
    [0.3, 0.1, 0.00, 0.5, 0.1],
    [0.5, 0.1, 0.1, 0.00, 0.3],
    [0.1, 0.2, 0.3, 0.4, 0.0]
]

In [3]:
N_STATUSES = 4

start_status_probs = [0.1, 0.1, 0.1, 0.7]

cab_status_matrix = [
    [0.00, 0.388, 0.409, 0.153],   # 0.00+0.40+0.40+0.18 = 0.98
    [0.184, 0.00, 0.409, 0.347],   # 0.20+0.40+0.38 = 0.98
    [0.30, 0.10, 0.00, 0.51],   # 0.30+0.10+0.58 = 0.98
    [0.53, 0.183, 0.137, 0.00]    # 0.60+0.20+0.18 = 0.98
]
death_status_probs  = [0.02, 0.03, 0.04, 0.1]
survive_status_probs= [0.03, 0.03, 0.05, 0.05]

In [ ]:
NUMBER_OF_SEQUENCES = 2

rng = np.random.default_rng()        # no seed = system entropy
model = TransitionModel(N_CABINETS, N_STATUSES)
model.build_from_matrices(start_status_probs, cab_status_matrix, death_status_probs, survive_status_probs, start_event_probs, cab_event_matrix)

generator = MedicalSequenceGenerator(model)
sequences: list[MedicalSequence] = generator.generate_many(NUMBER_OF_SEQUENCES)

START START
STATUS_4 CABINET_4
STATUS_5 CABINET_3
STATUS_2 CABINET_4
STATUS_3 CABINET_6
STATUS_4 CABINET_4
STATUS_3 CABINET_2
STATUS_4 CABINET_4
START START
STATUS_5 CABINET_4
STATUS_2 CABINET_5
STATUS_5 CABINET_6
STATUS_2 CABINET_5
STATUS_4 CABINET_6
STATUS_5 CABINET_4
STATUS_2 CABINET_5
STATUS_3 CABINET_2
STATUS_5 CABINET_3
STATUS_2 CABINET_2
STATUS_3 CABINET_3
STATUS_5 CABINET_4
STATUS_3 CABINET_5
STATUS_2 CABINET_6
STATUS_5 CABINET_4
STATUS_2 CABINET_5
STATUS_4 CABINET_4
STATUS_2 CABINET_2
STATUS_4 CABINET_6
STATUS_2 CABINET_4
STATUS_3 CABINET_2
STATUS_4 CABINET_4
STATUS_5 CABINET_2
STATUS_2 CABINET_3
STATUS_3 CABINET_4
STATUS_2 CABINET_5


In [5]:
def print_head_of_sequences(sequences: list[MedicalSequence]):
    print_size = min(len(sequences), 5)
    print("Token ID sequences:")
    for seq in sequences[:print_size]:
        seq.print_by_tokens()

    print("Named sequences:")
    for seq in sequences[:print_size]:
        seq.print_by_keys()

In [6]:
print_head_of_sequences(sequences)

Token ID sequences:
[(1, 1), (4, 4), (5, 3), (2, 4), (3, 6), (4, 4), (3, 2), (4, 4), (6, 2)]
[(1, 1), (5, 4), (2, 5), (5, 6), (2, 5), (4, 6), (5, 4), (2, 5), (3, 2), (5, 3), (2, 2), (3, 3), (5, 4), (3, 5), (2, 6), (5, 4), (2, 5), (4, 4), (2, 2), (4, 6), (2, 4), (3, 2), (4, 4), (5, 2), (2, 3), (3, 4), (2, 5), (7, 2)]
Named sequences:
[('START', 'START'), ('STATUS_4', 'CABINET_4'), ('STATUS_5', 'CABINET_3'), ('STATUS_2', 'CABINET_4'), ('STATUS_3', 'CABINET_6'), ('STATUS_4', 'CABINET_4'), ('STATUS_3', 'CABINET_2'), ('STATUS_4', 'CABINET_4'), ('DEATH', 'CABINET_2')]
[('START', 'START'), ('STATUS_5', 'CABINET_4'), ('STATUS_2', 'CABINET_5'), ('STATUS_5', 'CABINET_6'), ('STATUS_2', 'CABINET_5'), ('STATUS_4', 'CABINET_6'), ('STATUS_5', 'CABINET_4'), ('STATUS_2', 'CABINET_5'), ('STATUS_3', 'CABINET_2'), ('STATUS_5', 'CABINET_3'), ('STATUS_2', 'CABINET_2'), ('STATUS_3', 'CABINET_3'), ('STATUS_5', 'CABINET_4'), ('STATUS_3', 'CABINET_5'), ('STATUS_2', 'CABINET_6'), ('STATUS_5', 'CABINET_4'), ('STA

In [7]:
dataset = MedicalSequenceDataset(sequences)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)